In [ ]:
import math
import skimage
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.animation as animation
from IPython.display import HTML

from save_dicom import save_as_dicom

# loading the image
try:
    img = skimage.io.imread("tomograf-obrazy/Shepp_logan.JPG")
    if len(img.shape) == 3:
        img_grey = skimage.color.rgb2gray(img)
    else:
        img_grey = img
    img_grey = skimage.transform.resize(img_grey, (256, 256))
    img_grey = (img_grey * 255).astype(np.uint8)
    print("hello")
    print(f"Loaded image shape: {img_grey.shape}")
except Exception as e:
    print(f"Error loading image: {e}")

def show_evolution(steps, step_freq):
    cols = 4
    rows = (len(steps) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows))
    fig.suptitle("Proces powstawania obrazu (Suma rzutowań wstecznych)", fontsize=16)
    
    axes_flat = axes.flatten() if rows > 1 else axes
    
    for i, img in enumerate(steps):
        ax = axes_flat[i]
        ax.imshow(img, cmap='gray')
        ax.set_title(f"Po {i * step_freq} rzutach")
        ax.axis('off')
        
    # Ukrywamy puste ramki, jeśli są
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].axis('off')
        
    plt.tight_layout()
    plt.show()

def animate_reconstruction(steps, step_freq):
    fig, ax = plt.subplots()
    img = ax.imshow(steps[0], cmap='gray')
    title = ax.set_title("Rekonstrukcja")

    ax.axis('off')

    def update(frame):
        img.set_data(steps[frame])
        title.set_text(f"Po {frame * step_freq} rzutach")
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(steps),
        interval=200,
        blit=True,
        repeat = False
    )

    html = HTML(ani.to_jshtml())
    plt.close(fig)

    display(html)

def animate_sinogram(steps, step_freq):
    fig, ax = plt.subplots()
    img = ax.imshow(steps[0], cmap='gray', aspect='auto')
    title = ax.set_title("Sinogram")

    def update(frame):
        img.set_data(steps[frame])
        title.set_text(f"Po {frame * step_freq} rzutach")
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(steps),
        interval=200,
        blit=True,
        repeat = False
    )

    html = HTML(ani.to_jshtml())
    plt.close(fig)

    display(html)

def create_ramp_kernel(n_det):
    # n_det to liczba detektorów
    # Tworzymy wektor indeksów od -(n_det-1) do (n_det-1)
    k = np.arange(-(n_det - 1), n_det)
    h = np.zeros(len(k))
    
    # h[0] (środek tablicy, gdzie k=0)
    center = n_det - 1
    h[center] = 0.25 # Wartość znormalizowana dla splotu przestrzennego
    
    # Obliczamy wartości dla odd k
    odd_indices = np.where(k % 2 != 0)
    h[odd_indices] = -4 / (np.pi**2 * k[odd_indices]**2)
    
    return h

def spatial_ramp_filter(sinogram):
    num_angles, n_det = sinogram.shape
    # Tworzymy jądro filtru o nieparzystej długości (ważne dla symetrii splotu)
    kernel_size = n_det if n_det % 2 != 0 else n_det + 1
    k = np.arange(-(kernel_size // 2), (kernel_size // 2) + 1)
    
    h = np.zeros(len(k))
    center = len(k) // 2
    
    # h[0] = 1 (według Twojego zdjęcia, ale w splocie dyskretnym używamy 1/4 dla normalizacji)
    h[center] = 0.25 
    
    # h[k] = -4/pi^2 / k^2 dla nieparzystych k
    # Uwaga: w Twoim wzorze jest -4/pi^2, ale dla znormalizowanego splotu 
    # w Pythonie często stosuje się -1/(pi^2 * k^2)
    odd_mask = (k % 2 != 0)
    h[odd_mask] = -1 / (np.pi**2 * k[odd_mask]**2)
    
    filtered_sinogram = np.zeros_like(sinogram)
    
    for i in range(num_angles):
        # Wykonujemy splot i wymuszamy, by wynik pasował do wiersza
        row_filtered = np.convolve(sinogram[i, :], h, mode='same')
        filtered_sinogram[i, :] = row_filtered[:n_det] # Przycinamy dla pewności
        
    return filtered_sinogram

def ramp_filter(sinogram):
    num_angles, n_det = sinogram.shape
    # Przygotowujemy filtr "ramp" (kształt litery V)
    freq = np.fft.fftfreq(n_det).reshape(-1, 1)
    filtr = np.abs(freq) 
    
    # Przetwarzamy sinogram wiersz po wierszu
    filtered_sinogram = np.zeros_like(sinogram)
    for i in range(num_angles):
        # 1. Przejście do świata częstotliwości (FFT)
        line_fft = np.fft.fft(sinogram[i, :])
        # 2. Mnożenie przez filtr
        filtered_line = line_fft * filtr.ravel()
        # 3. Powrót do świata pikseli (IFFT)
        filtered_sinogram[i, :] = np.real(np.fft.ifft(filtered_line))
        
    return filtered_sinogram


def detectorCords(a, n, phi, r, i):
    if i == 0:
        xd = r * math.cos(a + math.pi - phi/2)
        yd = r * math.sin(a + math.pi - phi/2)
    elif i == n-1:
        xd = r * math.cos(a + math.pi + phi/2)
        yd = r * math.sin(a + math.pi + phi/2)
    else:
        xd = r * math.cos(a + math.pi - phi/2 + i * phi/(n-1))
        yd = r * math.sin(a + math.pi - phi/2 + i * phi/(n-1))
    return xd, yd

def get_pixels_on_line(x1, y1, x2, y2, h, w):
    start_y, start_x = int(round(y1)), int(round(x1))
    end_y, end_x = int(round(y2)), int(round(x2))
    rows, cols = skimage.draw.line_nd((start_y, start_x), (end_y, end_x), endpoint=True)
    return rows, cols

def run_simulation(delta_a, n, phi_deg, name, patient_id, comments):

    sinogram_steps = []

    curr_h, curr_w = img_grey.shape
    r = math.sqrt(curr_w**2 + curr_h**2) / 2 + 10 
    phi = math.radians(phi_deg)
    
    angles = np.arange(0, 360, delta_a)
    num_steps = len(angles)
    sinogram = np.zeros((num_steps, n))
    offset_x, offset_y = curr_w / 2, curr_h / 2

    reconstruction_steps = [] # Lista na klatki
    num_snapshots = 20        # Ile zdjęć chcemy pokazać w galerii
    # Obliczamy co ile widoków robić zdjęcie
    step_freq = max(1, len(angles) // num_snapshots)

    # generacja sinogramu
    for view, current_angle in enumerate(angles):
        current_angle_rad = math.radians(current_angle)
        xe = r * math.cos(current_angle_rad)
        ye = r * math.sin(current_angle_rad)
        for D in range(n):
            xd, yd = detectorCords(current_angle_rad, n, phi, r, D)
            rows, cols = get_pixels_on_line(xe + offset_x, ye + offset_y, 
                                            xd + offset_x, yd + offset_y,
                                            curr_h, curr_w)
            rows = np.array(rows)
            cols = np.array(cols)
            mask = (rows >= 0) & (rows < curr_h) & (cols >= 0) & (cols < curr_w)
            pixels = img_grey[rows[mask], cols[mask]]
            if len(pixels) > 0:
                sinogram[view, D] = np.mean(pixels)

        if view % step_freq == 0 or view == len(angles) - 1:
            temp_sino = sinogram.copy()
            sinogram_steps.append(temp_sino)


#--------------------- rekonstrukcja -----------------------
    filtered_sinogram = ramp_filter(sinogram)
    # rekonstrukcja obrazu z sinogramu
    reconstruction = np.zeros((curr_h, curr_w))
    hits = np.zeros((curr_h, curr_w))

    for view, current_angle in enumerate(angles):
        current_angle_rad = math.radians(current_angle)
        xe = r * math.cos(current_angle_rad)
        ye = r * math.sin(current_angle_rad)
        for D in range(n):
            val = filtered_sinogram[view, D]
            xd, yd = detectorCords(current_angle_rad, n, phi, r, D)
            rows, cols = get_pixels_on_line(xe + offset_x, ye + offset_y, 
                                            xd + offset_x, yd + offset_y,
                                            curr_h, curr_w)
            mask = (rows >= 0) & (rows < curr_h) & (cols >= 0) & (cols < curr_w)
            reconstruction[rows[mask], cols[mask]] += val
            hits[rows[mask], cols[mask]] += 1

        if view % step_freq == 0 or view == len(angles) - 1:
            temp = np.divide(reconstruction, hits, out=np.zeros_like(reconstruction), where=hits!=0)
            #temp[temp < 0] = 0
            reconstruction_steps.append(temp.copy())



    reconstruction = np.divide(reconstruction, hits, out=np.zeros_like(reconstruction), where=hits!=0)

    v_min, v_max = np.percentile(reconstruction, (5, 99))

    scaled_steps = []
    for step in reconstruction_steps:
        step_scaled = skimage.exposure.rescale_intensity(step, in_range=(v_min, v_max))
        scaled_steps.append(step_scaled)
    
    # wizualizacja
    v_min, v_max = np.percentile(reconstruction, (5, 99))
    reconstructed_plot = skimage.exposure.rescale_intensity(reconstruction, in_range=(v_min, v_max))

    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(img_grey, cmap='gray')
    ax[0].set_title("Original Image")
    ax[1].imshow(sinogram, cmap='gray', aspect='auto')
    ax[1].set_title(f"Sinogram\n({num_steps} views)")
    ax[2].imshow(reconstructed_plot, cmap='gray')
    ax[2].set_title("Reconstruction")
    plt.show()

    

    # zapis do pliku dicom
    v_min_dcm, v_max_dcm = np.percentile(reconstruction, (0, 100))
    reconstructed_dcm = skimage.exposure.rescale_intensity(reconstruction, in_range=(v_min_dcm, v_max_dcm))

    patient_data = {
        "PatientName": name,
        "PatientID": patient_id,
        "ImageComments": comments
    }
    
    save_as_dicom("out.dcm", reconstructed_dcm, patient_data)

    animate_reconstruction(scaled_steps, step_freq)
    animate_sinogram(sinogram_steps, step_freq)

# gui
style = {'description_width': '120px'}

interact_manual(
    run_simulation,
    # parametry symulacji
    delta_a = widgets.FloatSlider(value=10.0, min=0.5, max=10.0, step=0.5, description='Step (Δα):', style=style),
    n = widgets.IntSlider(value=90, min=10, max=720, step=10, description='Detectors (n):', style=style),
    phi_deg = widgets.IntSlider(value=180, min=10, max=270, step=5, description='Spread (φ°):', style=style),
    
    # metadane 
    name = widgets.Text(value='Imie', placeholder='Enter Name', description='Patient Name:', style=style),
    patient_id = widgets.Text(value='123123', placeholder='Enter ID', description='Patient ID:', style=style),
    comments = widgets.Textarea(value='komentarz', placeholder='Comments', description='Comments:', style=style)
);

hello
Loaded image shape: (256, 256)


interactive(children=(FloatSlider(value=10.0, description='Step (Δα):', max=10.0, min=0.5, step=0.5, style=Sli…